# 04 - Pose Inference
Run the trained pose model on all videos to extract keypoint coordinates.

In [ ]:
# ===== CONFIGURATION =====
GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"

DRIVE_ROOT = "/content/drive/My Drive/PigBehavior"
DRIVE_RAW_VIDEOS = f"{DRIVE_ROOT}/raw_videos"
DRIVE_MODELS = f"{DRIVE_ROOT}/trained_models"
DRIVE_POSE_OUTPUTS = f"{DRIVE_ROOT}/pose_outputs"

# Model file name (as saved from notebook 03)
MODEL_CHECKPOINT = "pose_model.ckpt"
CONFIDENCE_THRESHOLD = 0.5
DRIVE_FOLDER_ID = "1X_41ZW3HfwVeft2lPld3XNqXsdxRDIwb"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

In [ ]:
# Install Lightning Pose + inference dependencies
!pip install --quiet "lightning-pose[all]" opencv-python pandas numpy pyarrow

In [ ]:
from pathlib import Path

model_path = Path(DRIVE_MODELS) / MODEL_CHECKPOINT
if model_path.exists():
    print(f"Model found: {model_path} ({model_path.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"Model not found at {model_path}. Complete notebook 03 first.")

In [ ]:
from src.io.video_inventory import scan_videos, parse_camera_from_filename

df = scan_videos(DRIVE_RAW_VIDEOS)
print(f"Found {len(df)} videos to process")
df[["filename", "session", "camera", "frame_count", "duration_min"]]

In [ ]:
import torch
import pandas as pd
import numpy as np
import cv2
from pathlib import Path
from tqdm.notebook import tqdm

# Load the trained Lightning Pose model
# Lightning Pose provides a predict function; adapt based on your model type
from lightning_pose.utils.predict import predict_video

pose_output_dir = Path(DRIVE_POSE_OUTPUTS)
pose_output_dir.mkdir(parents=True, exist_ok=True)

KEYPOINT_NAMES = [
    "snout", "left_ear", "right_ear", "neck",
    "shoulders", "mid_back", "hip", "tail_base",
]

for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing videos"):
    video_path = Path(DRIVE_RAW_VIDEOS) / row["path"]
    session = row["session"]
    camera = row["camera"]
    stem = Path(row["filename"]).stem

    out_file = pose_output_dir / session / f"{stem}_cam{camera}_pose.parquet"
    if out_file.exists():
        print(f"Skipping {out_file.name} (already exists)")
        continue

    # Run inference with Lightning Pose
    # This generates a CSV/parquet output with keypoint predictions
    !python -m lightning_pose.predict \
        --checkpoint {model_path} \
        --video {video_path} \
        --output-dir /content/tmp_pose \
        --keypoint-names {' '.join(KEYPOINT_NAMES)}

    # Find the generated output file
    import glob
    generated = list(Path("/content/tmp_pose").glob("*.csv")) + list(Path("/content/tmp_pose").glob("*.parquet"))
    if generated:
        gen_df = pd.read_csv(str(generated[0])) if generated[0].suffix == ".csv" else pd.read_parquet(str(generated[0]))
        # Save to Drive
        out_file.parent.mkdir(parents=True, exist_ok=True)
        gen_df.to_parquet(str(out_file))
        print(f"Saved: {out_file} ({len(gen_df)} frames)")

print("\nPose inference complete!")

In [ ]:
# Verify outputs
output_files = list(pose_output_dir.rglob("*.parquet"))
print(f"Total pose output files: {len(output_files)}")
for f in output_files:
    print(f"  {f.relative_to(pose_output_dir)}")